## Imports

In [25]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error


## Load Dataset

In [26]:
df = pd.read_csv("../data/NFCS_2024_State_Data_250623.csv")

df.head()

,NFCSID,STATEQ,CENSUSDIV,CENSUSREG,A50A,A3Ar_w,A50B,A4A_new_w,A5_2015,A6,...,M6,M7,M8,M31,M50,M9,M10,wgt_n2,wgt_d2,wgt_s3
0,2024010001,36,3,2,2,3,9,1,6,1,...,1,3,98,98,98,1,2,1.153548,1.123949,0.859644
1,2024010002,48,9,4,1,6,6,1,5,5,...,1,3,98,3,3,1,2,1.398688,0.863440,0.975078
2,2024010003,38,9,4,1,6,6,1,4,1,...,1,3,1,98,98,2,2,1.398688,0.472645,0.893974
3,2024010004,48,9,4,2,6,12,1,7,1,...,1,3,1,4,98,1,2,1.250293,0.614156,0.778748
4,2024010005,44,7,3,2,6,12,2,7,4,...,1,3,2,3,2,1,1,1.250076,2.228957,0.783507


## Create Target Variable (Recommended Contribution)

In [27]:
def estimate_contribution(row):
    # Use only these to create target
    income = row["A8_2021"]
    risk = row["H30_3"] if "H30_3" in row else 0
    chronic = row.get("chronic_condition", 0)
    hospitalizations = row.get("hospitalizations", 0)

    # synthetic but independent rule
    base = 20
    base += (income * 5)
    base += (risk * 15)
    base += (chronic * 20)
    base += (hospitalizations * 25)

    return max(base, 10)

df["recommended_contribution"] = df.apply(estimate_contribution, axis=1)

df[["recommended_contribution"]].describe()

,recommended_contribution
count,25539.000000
mean,100.751791
std,207.007018
min,40.000000
25%,60.000000
50%,70.000000
75%,80.000000
max,1555.000000


## Feature Selection

In [28]:

feature_cols = [
    "PPAGE",
    "PPGENDER",
    "AA_new_w",
    "A5_2015",
    "A9",
    "J1",
    "J2",
    "J4",
    "J5",
    "H1",
    "F1",
    "G23",
    "wgt_n2"
]

# Notice: we removed A8_2021 and H30_3 (used in target generation)

feature_cols = [col for col in feature_cols if col in df.columns]

X = df[feature_cols]
y = df["recommended_contribution"]

X.head()

,A5_2015,A9,J1,J2,J4,J5,H1,F1,G23,wgt_n2
0,6,2,7,8,3,1,1,2,5,1.153548
1,5,8,9,6,3,1,1,3,1,1.398688
2,4,8,9,1,3,1,1,3,1,1.398688
3,7,1,4,5,2,1,1,1,1,1.250293
4,7,8,6,7,2,2,1,2,5,1.250076


## Preprocessing Pipeline

In [29]:
categorical = X.select_dtypes(include=["object"]).columns
numeric = [col for col in X.columns if col not in categorical]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
        ("num", "passthrough", numeric)
    ]
)

## Model Pipeline
- Random Forest

In [30]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ))
])

### Train/Test Split

In [31]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

### Train Model

In [32]:
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index([], dtype='object')),
                                                 ('num', 'passthrough',
                                                  ['A5_2015', 'A9', 'J1', 'J2',
                                                   'J4', 'J5', 'H1', 'F1',
                                                   'G23', 'wgt_n2'])])),
                ('regressor',
                 RandomForestRegressor(n_estimators=200, random_state=42))])

## Evaluate Performance

In [33]:
from sklearn.metrics import mean_absolute_error

val_pred = model.predict(X_val)
mae = mean_absolute_error(y_val, val_pred)

print(f"Validation MAE: {mae:.2f}")

Validation MAE: 64.10


On average, the model’s prediction is off by about $64 (or whatever unit your target represents).

In [34]:
joblib.dump(model, "../models/hsa_recommendation_model.pkl")

print("Model saved as hsa_recommendation_model.pkl")

Model saved as hsa_recommendation_model.pkl
